# Training


In [ ]:
import os
import math
import json
import time
import random
import glob
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import import_ipynb

In [ ]:
@dataclass
class TrainConfig:
    size: str = "tiny"
    batch_size: int = 32
    grad_accum: int = 2
    max_steps: int = 1048576
    warmup_steps: int = 2048
    peak_lr: float = 1.5e-3
    final_lr_ratio: float = 0.1
    weight_decay: float = 0.1
    betas: tuple = (0.9, 0.98)
    eps: float = 1e-6
    grad_clip: float = 1.0
    label_smoothing: float = 0.0
    text_ctx: int = 448
    eval_every: int = 2000
    checkpoint_every: int = 5000
    keep_checkpoints: int = 5
    seed: int = 20260315
    amp_dtype: str = "bfloat16"
    timestamps_prob: float = 0.5
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = TrainConfig()
torch.manual_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

In [ ]:
class ShardDataset(Dataset):
    def __init__(self, split_dir, tokenizer, frontend, text_ctx, timestamps_prob, seed):
        self.shards = sorted(Path(split_dir).glob("shard-*.jsonl"))
        self.tokenizer = tokenizer
        self.frontend = frontend
        self.text_ctx = text_ctx
        self.timestamps_prob = timestamps_prob
        self.rng = random.Random(seed)
        self.index = []
        for shard in self.shards:
            with open(shard) as f:
                for i, _ in enumerate(f):
                    self.index.append((shard, i))
        self._cache_path = None
        self._cache_audio = None
        self._cache_meta = None

    def __len__(self):
        return len(self.index)

    def _load_shard(self, shard):
        if self._cache_path == shard:
            return
        npz = np.load(str(shard).replace(".jsonl", ".npz"))
        self._cache_audio = [npz[k] for k in npz.files]
        with open(shard) as f:
            self._cache_meta = [json.loads(line) for line in f]
        self._cache_path = shard

    def __getitem__(self, idx):
        shard, offset = self.index[idx]
        self._load_shard(shard)
        meta = self._cache_meta[offset]
        wave = torch.from_numpy(self._cache_audio[offset].astype(np.float32))
        mel = self.frontend(wave)
        use_timestamps = self.rng.random() < self.timestamps_prob
        stamps = None
        if use_timestamps:
            stamps = [(0.0, min(meta["duration"], 29.98), meta["text"])]
        target = self.tokenizer.encode_target(meta, timestamps=stamps)
        target = target[: self.text_ctx]
        return mel, torch.tensor(target, dtype=torch.long)

def collate(batch, pad_id):
    mels = torch.stack([b[0] for b in batch])
    max_len = max(len(b[1]) for b in batch)
    tokens = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    loss_mask = torch.zeros((len(batch), max_len), dtype=torch.bool)
    for i, (_, t) in enumerate(batch):
        tokens[i, : len(t)] = t
        loss_mask[i, : len(t)] = True
    return mels, tokens, loss_mask

In [ ]:
def lr_at(step, cfg):
    if step < cfg.warmup_steps:
        return cfg.peak_lr * step / max(cfg.warmup_steps, 1)
    progress = (step - cfg.warmup_steps) / max(cfg.max_steps - cfg.warmup_steps, 1)
    floor = cfg.peak_lr * cfg.final_lr_ratio
    return floor + 0.5 * (cfg.peak_lr - floor) * (1 + math.cos(math.pi * min(progress, 1.0)))

xs = np.arange(0, 60000, 100)
ys = [lr_at(x, cfg) for x in xs]
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 2.5))
plt.plot(xs, ys)
plt.tight_layout()

In [ ]:
def build_optimizer(model, cfg):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() < 2 or "embedding" in name or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": cfg.weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=cfg.peak_lr,
        betas=cfg.betas,
        eps=cfg.eps,
    )

In [ ]:
def sequence_loss(logits, tokens, loss_mask, sot_len=3, label_smoothing=0.0):
    logits = logits[:, :-1]
    targets = tokens[:, 1:]
    mask = loss_mask[:, 1:].clone()
    mask[:, : sot_len - 1] = False
    losses = F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        targets.reshape(-1),
        reduction="none",
        label_smoothing=label_smoothing,
    ).view(targets.shape)
    losses = losses * mask
    return losses.sum() / mask.sum().clamp(min=1)

In [ ]:
class CheckpointManager:
    def __init__(self, run_dir, keep):
        self.run_dir = Path(run_dir)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        self.keep = keep

    def save(self, step, model, optimizer, cfg, metrics):
        path = self.run_dir / f"step-{step:08d}.pt"
        torch.save(
            {
                "step": step,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "config": asdict(cfg),
                "metrics": metrics,
                "rng": {
                    "torch": torch.get_rng_state(),
                    "numpy": np.random.get_state(),
                    "python": random.getstate(),
                },
            },
            path,
        )
        checkpoints = sorted(self.run_dir.glob("step-*.pt"))
        for old in checkpoints[: -self.keep]:
            old.unlink()
        return path

    def latest(self):
        checkpoints = sorted(self.run_dir.glob("step-*.pt"))
        return checkpoints[-1] if checkpoints else None

    def restore(self, model, optimizer):
        path = self.latest()
        if path is None:
            return 0
        state = torch.load(path, map_location="cpu", weights_only=False)
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        torch.set_rng_state(state["rng"]["torch"])
        np.random.set_state(state["rng"]["numpy"])
        random.setstate(state["rng"]["python"])
        return state["step"]

In [ ]:
class MetricsLog:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.f = open(self.path, "a")

    def write(self, step, **kv):
        row = {"step": step, "time": time.time()}
        row.update(kv)
        self.f.write(json.dumps(row) + "\n")
        self.f.flush()

In [ ]:
from model_architecture import ASRModel, CONFIGS
from tokenizer_training import MultitaskTokenizer, BPETokenizer, base_alphabet, merges
from audio_frontend import FrontendPipeline

bpe = BPETokenizer(base_alphabet, merges)
tokenizer = MultitaskTokenizer(bpe)
dims = CONFIGS[cfg.size]
dims.n_vocab = tokenizer.n_vocab
model = ASRModel(dims).to(cfg.device)
optimizer = build_optimizer(model, cfg)

train_ds = ShardDataset("data/work/train", tokenizer, FrontendPipeline(train=True, seed=cfg.seed),
                        cfg.text_ctx, cfg.timestamps_prob, cfg.seed)
dev_ds = ShardDataset("data/work/dev", tokenizer, FrontendPipeline(train=False),
                      cfg.text_ctx, 0.0, cfg.seed)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=8,
                          collate_fn=lambda b: collate(b, tokenizer.eot), drop_last=True,
                          persistent_workers=True, pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=4,
                        collate_fn=lambda b: collate(b, tokenizer.eot))
print(len(train_ds), len(dev_ds))

In [ ]:
@torch.no_grad()
def evaluate(model, loader, cfg, max_batches=200):
    model.eval()
    total, count = 0.0, 0
    for i, (mels, tokens, loss_mask) in enumerate(loader):
        if i >= max_batches:
            break
        mels = mels.to(cfg.device, non_blocking=True)
        tokens = tokens.to(cfg.device, non_blocking=True)
        loss_mask = loss_mask.to(cfg.device, non_blocking=True)
        with torch.autocast(cfg.device, dtype=getattr(torch, cfg.amp_dtype)):
            logits = model(mels, tokens)
            loss = sequence_loss(logits, tokens, loss_mask)
        total += float(loss)
        count += 1
    model.train()
    return total / max(count, 1)

In [ ]:
def train(model, optimizer, cfg, run_name):
    ckpt = CheckpointManager(f"runs/{run_name}", cfg.keep_checkpoints)
    log = MetricsLog(f"runs/{run_name}/metrics.jsonl")
    step = ckpt.restore(model, optimizer)
    scaler = torch.amp.GradScaler(enabled=cfg.amp_dtype == "float16")
    amp_dtype = getattr(torch, cfg.amp_dtype)
    model.train()
    accum_loss = 0.0
    data_iter = iter(train_loader)
    t0 = time.time()
    while step < cfg.max_steps:
        optimizer.zero_grad(set_to_none=True)
        for micro in range(cfg.grad_accum):
            try:
                mels, tokens, loss_mask = next(data_iter)
            except StopIteration:
                data_iter = iter(train_loader)
                mels, tokens, loss_mask = next(data_iter)
            mels = mels.to(cfg.device, non_blocking=True)
            tokens = tokens.to(cfg.device, non_blocking=True)
            loss_mask = loss_mask.to(cfg.device, non_blocking=True)
            with torch.autocast(cfg.device, dtype=amp_dtype):
                logits = model(mels, tokens)
                loss = sequence_loss(logits, tokens, loss_mask,
                                     label_smoothing=cfg.label_smoothing) / cfg.grad_accum
            scaler.scale(loss).backward()
            accum_loss += float(loss)
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        lr = lr_at(step, cfg)
        for group in optimizer.param_groups:
            group["lr"] = lr
        scaler.step(optimizer)
        scaler.update()
        step += 1
        if step % 50 == 0:
            elapsed = time.time() - t0
            log.write(step, loss=accum_loss / 50, lr=lr, grad_norm=float(grad_norm),
                      steps_per_sec=50 / elapsed)
            accum_loss = 0.0
            t0 = time.time()
        if step % cfg.eval_every == 0:
            dev_loss = evaluate(model, dev_loader, cfg)
            log.write(step, dev_loss=dev_loss)
        if step % cfg.checkpoint_every == 0:
            ckpt.save(step, model, optimizer, cfg, {"step": step})
    ckpt.save(step, model, optimizer, cfg, {"step": step, "final": True})
    return step

In [ ]:
final_step = train(model, optimizer, cfg, run_name=f"{cfg.size}-multitask-v1")
print(final_step)

In [ ]:
def plot_metrics(run_name):
    rows = []
    with open(f"runs/{run_name}/metrics.jsonl") as f:
        for line in f:
            rows.append(json.loads(line))
    train_pts = [(r["step"], r["loss"]) for r in rows if "loss" in r]
    dev_pts = [(r["step"], r["dev_loss"]) for r in rows if "dev_loss" in r]
    fig, ax = plt.subplots(figsize=(10, 3))
    if train_pts:
        ax.plot(*zip(*train_pts), label="train", alpha=0.6)
    if dev_pts:
        ax.plot(*zip(*dev_pts), label="dev", marker="o")
    ax.set_yscale("log")
    ax.set_xlabel("step")
    ax.set_ylabel("loss")
    ax.legend()
    fig.tight_layout()

plot_metrics(f"{cfg.size}-multitask-v1")